In [9]:
import requests
import pandas as pd
import pulp

In [10]:
df = pd.read_csv("FC26_20250921.csv", low_memory=False)

# Attribute columns used for scoring
ATTR_COLS = [
    'movement_acceleration', 'movement_sprint_speed', 'movement_agility',
    'movement_reactions', 'movement_balance',
    'attacking_finishing', 'attacking_volleys', 'attacking_heading_accuracy',
    'attacking_short_passing', 'attacking_crossing',
    'power_shot_power', 'power_long_shots', 'power_jumping',
    'power_stamina', 'power_strength',
    'skill_dribbling', 'skill_ball_control', 'skill_long_passing',
    'mentality_positioning', 'mentality_composure', 'mentality_vision',
    'mentality_interceptions', 'mentality_aggression',
    'defending_marking_awareness', 'defending_standing_tackle', 'defending_sliding_tackle',
    'physic',
    'goalkeeping_diving', 'goalkeeping_handling', 'goalkeeping_kicking',
    'goalkeeping_positioning', 'goalkeeping_reflexes', 'goalkeeping_speed',
]

# Ensure all attribute columns are numeric; fill NaN with 0
df[ATTR_COLS] = df[ATTR_COLS].apply(pd.to_numeric, errors='coerce').fillna(0)

# Ensure value_eur is numeric; fill missing with 0
df['value_eur'] = pd.to_numeric(df['value_eur'], errors='coerce').fillna(0)

# Drop players with no market value
df = df[df['value_eur'] != 0].reset_index(drop=True)

print(f"Loaded {len(df):,} players")

Loaded 18,297 players


In [11]:
def _avg(row, cols):
    """Return the mean of the given columns for a single row."""
    return sum(row[c] for c in cols) / len(cols)


def forward_score(row):
    """
    Score a player as a Forward (ST / CF / LW / RW).
    Rewards finishing, pace, dribbling, and clinical mentality.
    """
    finishing = _avg(row, ['attacking_finishing', 'attacking_volleys',
                           'power_shot_power', 'power_long_shots', 'mentality_positioning'])
    pace       = _avg(row, ['movement_acceleration', 'movement_sprint_speed'])
    dribbling  = _avg(row, ['skill_dribbling', 'skill_ball_control', 'movement_agility'])
    physical   = _avg(row, ['attacking_heading_accuracy', 'power_jumping', 'power_strength'])
    mental     = _avg(row, ['mentality_composure', 'movement_reactions'])
    return round(finishing * 0.30 + pace * 0.20 + dribbling * 0.20
                 + physical * 0.15 + mental * 0.15, 1)


def midfielder_score(row):
    """
    Score a player as a Midfielder (CM / CAM / CDM / LM / RM).
    Rewards passing, vision, dribbling, stamina, and defensive awareness.
    """
    passing    = _avg(row, ['attacking_short_passing', 'skill_long_passing',
                            'mentality_vision', 'attacking_crossing'])
    dribbling  = _avg(row, ['skill_dribbling', 'skill_ball_control', 'movement_agility'])
    physical   = _avg(row, ['power_stamina', 'physic'])
    defending  = _avg(row, ['mentality_interceptions', 'defending_marking_awareness'])
    pace_react = _avg(row, ['movement_acceleration', 'movement_reactions'])
    return round(passing * 0.30 + dribbling * 0.20 + physical * 0.15
                 + defending * 0.15 + pace_react * 0.20, 1)


def defender_score(row):
    """
    Score a player as a Defender (CB / LB / RB / LWB / RWB).
    Rewards tackling, marking, physicality, and composure under pressure.
    """
    defending  = _avg(row, ['defending_marking_awareness', 'defending_standing_tackle',
                            'defending_sliding_tackle', 'mentality_interceptions'])
    physical   = _avg(row, ['power_strength', 'power_jumping',
                            'attacking_heading_accuracy', 'physic'])
    mental     = _avg(row, ['mentality_aggression', 'mentality_composure', 'movement_reactions'])
    pace       = _avg(row, ['movement_acceleration', 'movement_sprint_speed'])
    return round(defending * 0.40 + physical * 0.25 + mental * 0.20 + pace * 0.15, 1)


def goalkeeper_score(row):
    """
    Score a player as a Goalkeeper (GK).
    Based entirely on goalkeeping-specific attributes.
    """
    shot_stop = _avg(row, ['goalkeeping_diving', 'goalkeeping_reflexes', 'goalkeeping_positioning'])
    distribut = _avg(row, ['goalkeeping_handling', 'goalkeeping_kicking'])
    mobility  = row['goalkeeping_speed']
    return round(shot_stop * 0.60 + distribut * 0.30 + mobility * 0.10, 1)


print("Scoring functions defined.")

Scoring functions defined.


In [12]:
df['forward_score']    = df.apply(forward_score, axis=1)
df['midfielder_score'] = df.apply(midfielder_score, axis=1)
df['defender_score']   = df.apply(defender_score, axis=1)
df['goalkeeper_score'] = df.apply(goalkeeper_score, axis=1)

print("Scores added. Score ranges:")
for col in ['forward_score', 'midfielder_score', 'defender_score', 'goalkeeper_score']:
    print(f"  {col}: {df[col].min():.1f} – {df[col].max():.1f}")

Scores added. Score ranges:
  forward_score: 19.4 – 90.4
  midfielder_score: 16.4 – 85.4
  defender_score: 17.8 – 86.8
  goalkeeper_score: 1.8 – 84.2


In [13]:
DISPLAY_COLS = ['short_name', 'player_positions', 'overall', 'value_eur',
                'forward_score', 'midfielder_score', 'defender_score', 'goalkeeper_score']

print("=== Top 10 Forwards ===")
display(df[DISPLAY_COLS].sort_values('forward_score', ascending=False).head(10).reset_index(drop=True))

print("\n=== Top 10 Midfielders ===")
display(df[DISPLAY_COLS].sort_values('midfielder_score', ascending=False).head(10).reset_index(drop=True))

print("\n=== Top 10 Defenders ===")
display(df[DISPLAY_COLS].sort_values('defender_score', ascending=False).head(10).reset_index(drop=True))

print("\n=== Top 10 Goalkeepers ===")
display(df[DISPLAY_COLS].sort_values('goalkeeper_score', ascending=False).head(10).reset_index(drop=True))

=== Top 10 Forwards ===


,short_name,player_positions,overall,value_eur,forward_score,midfielder_score,defender_score,goalkeeper_score
0,K. Mbappé,"ST, LW, LM",91,173500000,90.4,78.2,63.6,7.8
1,O. Dembélé,"ST, RW, CAM",90,122500000,88.2,80.0,66.2,8.8
2,E. Haaland,ST,90,157000000,86.8,71.7,68.6,9.1
3,V. Osimhen,ST,87,100000000,86.2,71.1,65.6,9.4
4,L. Martínez,ST,88,99000000,86.2,76.3,70.1,8.8
5,M. Salah,"RM, RW",91,82000000,86.0,81.0,65.7,11.2
6,V. Gyökeres,ST,87,93000000,85.8,73.8,64.3,10.0
7,J. Alvarez,ST,87,107000000,85.5,79.6,71.1,8.1
8,J. Bellingham,"CAM, CM",90,174500000,85.1,85.3,81.6,8.5
9,A. Isak,ST,88,111000000,84.9,72.7,62.0,7.2



=== Top 10 Midfielders ===


,short_name,player_positions,overall,value_eur,forward_score,midfielder_score,defender_score,goalkeeper_score
0,J. Kimmich,"CDM, RB, CM",89,86000000,77.8,85.4,80.8,10.9
1,J. Bellingham,"CAM, CM",90,174500000,85.1,85.3,81.6,8.5
2,F. Valverde,"CM, CDM, RB",89,120500000,84.1,85.3,83.7,8.2
3,Pedri,"CM, CDM, CAM",89,149500000,78.4,85.1,77.7,8.3
4,N. Barella,CM,87,79500000,80.9,85.0,80.0,9.9
5,A. Hakimi,"RB, RM",89,111000000,83.2,84.2,82.8,8.1
6,F. de Jong,"CM, CDM",87,79500000,79.9,84.0,79.3,8.6
7,Vitinha,"CM, CDM, CAM",89,128500000,77.5,84.0,73.8,7.5
8,Rodri,"CDM, CM",90,102000000,79.4,83.7,82.7,8.9
9,T. Reijnders,"CM, CDM, CAM",86,79500000,79.6,82.9,77.2,8.7



=== Top 10 Defenders ===


,short_name,player_positions,overall,value_eur,forward_score,midfielder_score,defender_score,goalkeeper_score
0,V. van Dijk,CB,90,57000000,72.1,76.2,86.8,10.4
1,Marquinhos,CB,87,54500000,72.0,79.3,85.1,8.5
2,J. Koundé,"RB, CB, RM",87,85500000,71.9,81.0,84.5,10.3
3,W. Saliba,CB,87,92000000,66.8,75.2,84.5,7.6
4,A. Rüdiger,CB,86,44000000,71.2,75.0,84.4,10.6
5,W. Pacho,CB,86,83000000,63.6,72.0,84.3,10.0
6,A. Bastoni,CB,87,87000000,69.0,79.3,84.2,8.9
7,Bremer,CB,85,51000000,69.4,69.9,83.9,9.0
8,Ibañez,CB,82,33500000,64.0,67.8,83.8,8.6
9,I. Konaté,CB,86,70000000,64.4,72.7,83.7,9.0



=== Top 10 Goalkeepers ===


,short_name,player_positions,overall,value_eur,forward_score,midfielder_score,defender_score,goalkeeper_score
0,Alisson,GK,89,51000000,46.6,41.4,38.1,84.2
1,David Raya,GK,87,54500000,46.4,44.4,35.5,83.5
2,M. Maignan,GK,87,61000000,49.7,45.8,42.0,82.8
3,Ederson,GK,85,28500000,49.9,47.8,40.7,82.2
4,T. Courtois,GK,89,33500000,41.9,34.5,35.2,81.9
5,Y. Sommer,GK,87,9000000,41.8,38.1,34.8,81.8
6,G. Donnarumma,GK,89,97000000,43.0,38.6,37.0,81.5
7,J. Oblak,GK,88,45000000,42.7,39.0,38.3,81.4
8,M. ter Stegen,GK,86,22000000,42.1,40.9,37.3,81.2
9,E. Martínez,GK,85,26500000,46.8,43.8,41.1,80.8


In [14]:
# Map first listed position token to one of four squad categories
POSITION_MAP = {
    'GK':  'GKP',
    'CB':  'DEF', 'LB':  'DEF', 'RB':  'DEF', 'LWB': 'DEF', 'RWB': 'DEF',
    'CM':  'MID', 'CAM': 'MID', 'CDM': 'MID', 'LM':  'MID', 'RM':  'MID',
    'ST':  'FWD', 'CF':  'FWD', 'LW':  'FWD', 'RW':  'FWD', 'LF':  'FWD', 'RF':  'FWD',
}

def classify_position(pos_str):
    first = str(pos_str).split(',')[0].strip()
    return POSITION_MAP.get(first, 'MID')  # default unrecognised tokens to MID

df['category'] = df['player_positions'].apply(classify_position)

# Each player's rating = their score for the category they play in
SCORE_COL = {
    'GKP': 'goalkeeper_score',
    'DEF': 'defender_score',
    'MID': 'midfielder_score',
    'FWD': 'forward_score',
}
df['rating'] = df.apply(lambda r: r[SCORE_COL[r['category']]], axis=1)

# Deduplicate to one row per player (keep highest-rated row)
opt_df = (df.sort_values('rating', ascending=False)
            .drop_duplicates('player_id')
            .reset_index(drop=True))

print("Player pool by category:")
print(opt_df['category'].value_counts().to_string())

Player pool by category:
category
MID    6827
DEF    6083
FWD    3337
GKP    2050


In [15]:
# ── Optimization inputs ───────────────────────────────────────────────────────
names     = list(opt_df['short_name'])
ratings   = list(opt_df['rating'])
prices    = list(opt_df['value_eur'])
positions = list(opt_df['category'])
clubs     = list(opt_df['club_name'])
n         = len(opt_df)

BUDGET = 1_000_000_000  # €1B

# ── Model ─────────────────────────────────────────────────────────────────────
model = pulp.LpProblem("FC26_Squad_Optimizer", pulp.LpMaximize)
x = [pulp.LpVariable(f"x{i}", cat='Binary') for i in range(n)]

# Objective: maximise total positional rating
model += pulp.lpSum(x[i] * ratings[i] for i in range(n)), "Total_Rating"

# Budget constraint
model += pulp.lpSum(x[i] * prices[i] for i in range(n)) <= BUDGET, "Budget"

# Formation constraints
model += pulp.lpSum(x[i] for i in range(n) if positions[i] == 'GKP') == 1, "GK_count"
model += pulp.lpSum(x[i] for i in range(n) if positions[i] == 'DEF') >= 3, "DEF_min"
model += pulp.lpSum(x[i] for i in range(n) if positions[i] == 'MID') >= 2, "MID_min"
model += pulp.lpSum(x[i] for i in range(n) if positions[i] == 'FWD') >= 1, "FWD_min"

# Total squad size
model += pulp.lpSum(x) == 11, "Total_players"


# ── Solve ─────────────────────────────────────────────────────────────────────
model.solve(pulp.PULP_CBC_CMD(msg=0))
print("Status:", pulp.LpStatus[model.status])

Status: Optimal


In [16]:
selected_idx = [i for i in range(n) if x[i].value() == 1]
squad = opt_df.loc[selected_idx, ['short_name', 'club_name', 'player_positions',
                                   'category', 'rating', 'value_eur']].copy()

counts = squad['category'].value_counts()
formation = f"1 GK · {counts.get('DEF', 0)} DEF · {counts.get('MID', 0)} MID · {counts.get('FWD', 0)} FWD"
print(f"============ Optimal Squad ({formation}) ============\n")
for cat in ['GKP', 'DEF', 'MID', 'FWD']:
    for _, row in squad[squad['category'] == cat].iterrows():
        print(f"  {row['category']}  {row['short_name']:<22} {row['club_name']:<28}"
              f"rating={row['rating']:.1f}   €{row['value_eur']/1e6:.1f}M")
    print()

total_cost   = squad['value_eur'].sum()
total_rating = squad['rating'].sum()
print(f"Total cost:    €{total_cost/1e6:.1f}M  /  budget €{BUDGET/1e6:.0f}M")
print(f"Total rating:  {total_rating:.1f}")

============ Optimal Squad (1 GK · 3 DEF · 2 MID · 5 FWD) ============

  GKP  Alisson                Liverpool                   rating=84.2   €51.0M

  DEF  V. van Dijk            Liverpool                   rating=86.8   €57.0M
  DEF  Marquinhos             Paris Saint-Germain         rating=85.1   €54.5M
  DEF  A. Rüdiger             Real Madrid                 rating=84.4   €44.0M

  MID  J. Kimmich             FC Bayern München           rating=85.4   €86.0M
  MID  N. Barella             Inter                       rating=85.0   €79.5M

  FWD  K. Mbappé              Real Madrid                 rating=90.4   €173.5M
  FWD  O. Dembélé             Paris Saint-Germain         rating=88.2   €122.5M
  FWD  V. Osimhen             Galatasaray SK              rating=86.2   €100.0M
  FWD  L. Martínez            Inter                       rating=86.2   €99.0M
  FWD  V. Gyökeres            Arsenal                     rating=85.8   €93.0M

Total cost:    €960.0M  /  budget €1000M
Total ratin